# VAR(12) Autocorrelation Testing
This notebook tests for autocorrelation in the residuals of VAR(12) models using the Lagrange Multiplier (LM) test (`statsmodels.stats.diagnostic.acorr_lm`).
We analyze each Case-Shiller index area separately, checking test lags from 1 to 12.

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_lm

# Load the data
file_path = 'data_csv/var_data_1987_2007.csv'
df = pd.read_csv(file_path, parse_dates=['Unnamed: 0'], index_col='Unnamed: 0')
df.index.name = 'Date'

# Display first few rows
print(df.head())

                cs       cs10     pce   ffr  unemp         sf         ce  \
Date                                                                       
1987-01-01  63.733  63.103814  51.894  6.43    6.6  46.955792  54.057393   
1987-02-01  64.132  63.804972  52.076  6.10    6.6  47.302675  54.229218   
1987-03-01  64.468  64.425674  52.214  6.13    6.6  47.840213  54.365756   
1987-04-01  64.972  65.063393  52.409  6.37    6.3  47.984058  54.155370   
1987-05-01  65.547  65.815103  52.542  6.85    6.3  48.305065  54.726192   

                   cr         lv         mi  
Date                                         
1987-01-01  63.388189  66.358296  68.499290  
1987-02-01  63.936295  67.030115  68.764210  
1987-03-01  64.171042  67.335161  69.231280  
1987-04-01  64.813123  67.882287  69.198943  
1987-05-01  65.181595  67.898858  69.457298  


In [38]:
# List of Case-Shiller index columns
cs_cols = ['cs', 'cs10', 'sf', 'lv', 'ce', 'cr']

# Apply Log Transformations
df['log_pce'] = np.log(df['pce'])

for col in cs_cols:
    df[f'log_{col}'] = np.log(df[col])

# Check the transformed data
print(df[['unemp', 'log_pce', 'ffr'] + [f'log_{col}' for col in cs_cols]].head())

            unemp   log_pce   ffr    log_cs  log_cs10    log_sf    log_lv  \
Date                                                                        
1987-01-01    6.6  3.949203  6.43  4.154702  4.144781  3.849207  4.195069   
1987-02-01    6.6  3.952704  6.10  4.160943  4.155831  3.856567  4.205142   
1987-03-01    6.6  3.955351  6.13  4.166169  4.165512  3.867867  4.209683   
1987-04-01    6.3  3.959078  6.37  4.173956  4.175362  3.870869  4.217775   
1987-05-01    6.3  3.961613  6.85  4.182767  4.186849  3.877536  4.218019   

              log_ce    log_cr  
Date                            
1987-01-01  3.990046  4.149278  
1987-02-01  3.993220  4.157887  
1987-03-01  3.995734  4.161552  
1987-04-01  3.991857  4.171508  
1987-05-01  4.002342  4.177177  


In [39]:
from IPython.display import display, Markdown
from scipy import stats
from statsmodels.tsa.tsatools import lagmat

def calculate_multivariate_lm_test(results, max_lag):
    """
    Calculates the Multivariate Lagrange Multiplier (LM) test for autocorrelation 
    at each lag j, similar to Stata's varlmar.
    
    Tests H0: No autocorrelation at lag j.
    Statistic is based on the Likelihood Ratio of an auxiliary regression.
    """
    U = results.resid
    
    # Reconstruct X (design matrix)
    Y = results.model.endog
    p = results.k_ar
    
    # Create lags of Y (original regressors)
    # trim='both' cuts the first p observations to match residuals
    X_constructed = lagmat(Y, p, trim='both', original='ex', use_pandas=False)
    
    # Add constant if model has it (k_trend > 0)
    # Note: statsmodels VAR usually puts constant first if 'c'
    if results.k_trend > 0:
        X_constructed = sm.add_constant(X_constructed, prepend=True)
        
    X = X_constructed
    
    n_obs, k_vars = U.shape
    
    lm_results = []
    
    for lag in range(1, max_lag + 1):
        # Define sample for auxiliary regression (must lose 'lag' observations)
        # U_curr (target) starts from 'lag'
        U_curr = U[lag:]
        
        # Regressors: Original X (aligned) and Lagged U
        X_curr = X[lag:]
        U_lagged = U[:-lag]
        
        # Combine regressors for Unrestricted model
        # Stack horizontally
        X_unrestricted = np.hstack([X_curr, U_lagged])
        
        # Fit Unrestricted Model (U_curr ~ X_curr + U_lagged)
        # We use OLS formula: B = (X'X)^-1 X'Y
        # Residuals E = Y - X B
        # Sigma = E'E / T
        
        # Unrestricted
        try:
            B_ur = np.linalg.lstsq(X_unrestricted, U_curr, rcond=None)[0]
            E_ur = U_curr - X_unrestricted @ B_ur
            Sigma_ur = (E_ur.T @ E_ur) / len(U_curr)
            
            # Restricted Model (U_curr ~ X_curr)
            B_r = np.linalg.lstsq(X_curr, U_curr, rcond=None)[0]
            E_r = U_curr - X_curr @ B_r
            Sigma_r = (E_r.T @ E_r) / len(U_curr)
            
            # Calculate LR Statistic
            # LR = T * (ln|Sigma_r| - ln|Sigma_ur|)
            # This is asymptotically equivalent to LM
            sign_r, logdet_r = np.linalg.slogdet(Sigma_r)
            sign_ur, logdet_ur = np.linalg.slogdet(Sigma_ur)
            
            lr_stat = len(U_curr) * (logdet_r - logdet_ur)
            
            # Degrees of freedom = K^2 (since we test K*K coefficients of U_lagged)
            df = k_vars**2
            p_value = 1 - stats.chi2.cdf(lr_stat, df)
            
            lm_results.append({
                'Lag': lag,
                'Chi2': lr_stat,
                'df': df,
                'Prob > Chi2': p_value
            })
            
        except Exception as e:
            print(f"Error at lag {lag}: {e}")
            lm_results.append({
                'Lag': lag,
                'Chi2': np.nan,
                'df': k_vars**2,
                'Prob > Chi2': np.nan
            })

    return pd.DataFrame(lm_results).set_index('Lag')

def test_var12_autocorrelation(df, cs_col):
    """
    Fits a VAR(12) model and performs the Multivariate LM test for lags 1-12.
    """
    # Define variables
    var_vars = ['unemp', 'log_pce', 'ffr', f'log_{cs_col}']
    data = df[var_vars].dropna()
    
    # Fit VAR(12)
    model = VAR(data)
    results = model.fit(12)
    
    # Perform LM Test
    lm_df = calculate_multivariate_lm_test(results, max_lag=12)
    
    # Display results
    display(Markdown(f"### Analysis for **{cs_col}** (VAR(12)) - Multivariate LM Test"))
    display(lm_df.round(4))
    
    # Check significance
    if (lm_df['Prob > Chi2'] < 0.05).any():
        display(Markdown("**WARNING: Significant autocorrelation detected (p < 0.05) at some lags.**"))
    else:
        display(Markdown("No significant autocorrelation detected."))

In [40]:
# Run analysis for all areas
for col in cs_cols:
    test_var12_autocorrelation(df, col)

c:\Users\tuoma\Desktop\gradu_python\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


### Analysis for **cs** (VAR(12)) - Multivariate LM Test

,Chi2,df,Prob > Chi2
Lag,,,
1,23.3704,16,0.1042
2,60.3620,16,0.0000
3,42.6312,16,0.0003
4,38.1756,16,0.0014
5,42.7645,16,0.0003
6,45.0224,16,0.0001
7,42.2818,16,0.0004
8,24.2148,16,0.0849
9,26.6699,16,0.0453


**WARNING: Significant autocorrelation detected (p < 0.05) at some lags.**

c:\Users\tuoma\Desktop\gradu_python\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


### Analysis for **cs10** (VAR(12)) - Multivariate LM Test

... (14 outputs left out)

c:\Users\tuoma\Desktop\gradu_python\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


### Analysis for **cr** (VAR(12)) - Multivariate LM Test

,Chi2,df,Prob > Chi2
Lag,,,
1,25.4666,16,0.0620
2,41.2772,16,0.0005
3,44.5676,16,0.0002
4,20.6357,16,0.1929
5,23.0435,16,0.1126
6,20.9321,16,0.1811
7,46.8125,16,0.0001
8,34.8671,16,0.0041
9,13.2674,16,0.6531


**WARNING: Significant autocorrelation detected (p < 0.05) at some lags.**